1. importing dataset and config setting

In [ ]:
%pip install -q datasets

In [ ]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset, concatenate_datasets
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD

warnings.filterwarnings("ignore")


# ============================================================
# Project configuration
# ============================================================

CONFIG = {
    "dataset_name": "SHULAMITSHARABANI/consumer-complaints-pro",
    "random_state": 42,
    "min_words": 10,
    "max_sample_size": 10_000,
    "min_df": 5,
    "max_df": 0.90,
    "max_features": 8_000,
    "ngram_range": (1, 2),
    "k_min": 2,
    "k_max": 10,
    "silhouette_sample_size": 2_000,
    "top_terms_per_cluster": 15,
    "representative_documents": 3,
}


# ============================================================
# Project directories
# ============================================================

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

for directory in [
    DATA_DIR,
    OUTPUT_DIR,
    FIGURE_DIR,
    TABLE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


RAW_DATA_PATH = DATA_DIR / "complaints_raw.csv"
CLEAN_DATA_PATH = DATA_DIR / "complaints_clean.csv"

print("Project configuration completed.")

## Cell 3 — Downloading and Inspecting the Dataset

1. **Display the starting message:**  
   The `print` statement informs the user that the dataset download process has started.

2. **Connect to the dataset:**  
   The `load_dataset()` function accesses the consumer financial complaints dataset from Hugging Face.

3. **Select the dataset split:**  
   The argument `split="train"` specifies that records should be read from the dataset's main training split.

4. **Read the data in streaming mode:**  
   The option `streaming=True` reads the dataset gradually instead of downloading the entire large dataset at once.  
   It is similar to watching a movie online: you can start watching immediately without downloading the whole movie first.

5. **Create an empty list:**  
   The `records` variable is created to store the rows retrieved from the data stream.

6. **Iterate through the records:**  
   The `for` loop reads each record one by one together with its row number.

7. **Add records to the list:**  
   The `append()` method adds each new record to the `records` list.

8. **Limit the amount of data:**  
   The `if` condition stops the loop with `break` after the specified number of records has been collected.

9. **Check whether data was downloaded:**  
   If no records were retrieved, the notebook stops with a `RuntimeError`.

10. **Convert records into a DataFrame:**  
    The list of records is converted into a raw pandas DataFrame called `df_raw`.

11. **Display dataset dimensions and columns:**  
    The `shape` attribute shows the number of rows and columns, while `columns` displays the available column names.

12. **Display sample records:**  
    The command `display(df_raw.head())` shows the first five rows so that the dataset structure and content can be inspected.

In [ ]:
# ============================================================
# Download a reproducible sample from a healthy CFPB dataset
# ============================================================

DATASET_NAME = "Mouwiya/cfpb-consumer-complaints"

print("Downloading CFPB consumer complaints from Hugging Face...")

dataset_object = load_dataset(
    DATASET_NAME,
    split="train",
    streaming=True,
)

# Convert only a limited number of rows to avoid downloading
# the complete multi-million-row dataset.
records = []

download_limit = 50_000

for row_number, row in enumerate(dataset_object):

    records.append(row)

    if row_number + 1 >= download_limit:
        break


df_raw = pd.DataFrame(records)


print("\nDataset downloaded successfully.")
print("Raw downloaded shape:", df_raw.shape)

print("\nAvailable columns:")
for column in df_raw.columns:
    print("-", column)

display(df_raw.head())

## Cell 4 — Saving Raw Data and Detecting the Text Column

> **Main purpose:** Save the original dataset and automatically identify the column containing the complaint text.

### 1. Save the raw dataset

Cell 4 first saves the raw DataFrame, `df_raw`, as a CSV file using `to_csv()` and the path stored in `RAW_DATA_PATH`.

The argument `index=False` prevents pandas from writing the internal row numbers as an unnecessary extra column.

> **Why this matters:**  
> Saving the raw version preserves the original downloaded data before cleaning, which is useful for reproducibility, comparison, and debugging.

### 2. Search for the text column

The code defines several possible names for the complaint narrative column because different dataset versions may use names such as:

- `Consumer complaint narrative`
- `complaint_what_happened`
- `narrative`
- `text`

The `next()` function searches for the first exact match among the actual columns in `df_raw.columns`.

If no exact match is found, the code returns `None` and uses a flexible fallback method.

### 3. Use a flexible fallback method

The fallback method checks every column name, converts it to lowercase with `lower()`, replaces underscores with spaces using `replace("_", " ")`, and searches for keywords such as `narrative` or `complaint text`.

### 4. Store the detected name in `TEXT_COLUMN`

Once the correct column is found, its original name is stored in a standard variable called `TEXT_COLUMN`.

> **Important benefit:**  
> The rest of the notebook always uses `TEXT_COLUMN` instead of repeatedly using the dataset’s original column name. This makes the pipeline more flexible and reusable because later cells do not need to change if another dataset version uses a different name for the same text field.

### 5. Validate the result

If no suitable text column is found after both methods, the notebook stops with a clear `ValueError` and displays all available column names for debugging.

Finally, the detected text-column name and the path of the saved raw CSV file are printed.

In [ ]:
# ============================================================
# Save raw data
# ============================================================

df_raw.to_csv(
    RAW_DATA_PATH,
    index=False,
)


# ============================================================
# Automatically detect the complaint narrative column
# ============================================================

possible_text_columns = [
    "Consumer complaint narrative",
    "consumer_complaint_narrative",
    "complaint_what_happened",
    "Complaint narrative",
    "complaint_narrative",
    "consumer narrative",
    "narrative",
    "complaint_text",
    "text",
]


TEXT_COLUMN = next(
    (
        column
        for column in possible_text_columns
        if column in df_raw.columns
    ),
    None,
)


# Flexible fallback based on column names
if TEXT_COLUMN is None:

    for column in df_raw.columns:

        normalized_name = column.lower().replace(
            "_",
            " "
        )

        if (
            "narrative" in normalized_name
            or "complaint text" in normalized_name
        ):
            TEXT_COLUMN = column
            break


if TEXT_COLUMN is None:

    raise ValueError(
        "No complaint narrative column was detected. "
        f"Available columns: {df_raw.columns.tolist()}"
    )


print("Detected text column:")
print(TEXT_COLUMN)

print(f"\nRaw data saved to: {RAW_DATA_PATH}")

## Cell 5 — Detecting Metadata Columns

The `detect_column()` function is defined once and then called three times.  
Each call searches `df_raw` for a different type of metadata column.  
The first call detects the product column, the second detects the issue column, and the third detects the company column.  
The detected column names are stored in `PRODUCT_COLUMN`, `ISSUE_COLUMN`, and `COMPANY_COLUMN`.  
These metadata columns are not used to train K-means; they are used later to interpret the resulting clusters.

In [ ]:
# ============================================================
# Detect optional metadata columns
# ============================================================

def detect_column(dataframe, candidates):
    """
    Return the first matching column from a list of candidates.
    """

    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate

    return None


PRODUCT_COLUMN = detect_column(
    df_raw,
    [
        "Product",
        "product",
        "Category",
        "category",
    ],
)


ISSUE_COLUMN = detect_column(
    df_raw,
    [
        "Issue",
        "issue",
        "Complaint issue",
        "complaint_issue",
    ],
)


COMPANY_COLUMN = detect_column(
    df_raw,
    [
        "Company",
        "company",
    ],
)


print("Detected metadata columns:")
print("Product column:", PRODUCT_COLUMN)
print("Issue column:", ISSUE_COLUMN)
print("Company column:", COMPANY_COLUMN)

## Cell 6 — Initial Data Cleaning

Cell 6 creates an independent working copy of `df_raw` so the original downloaded data remains unchanged. It stores the initial number of rows, removes records with missing or empty complaint narratives, converts the text column to string format, and removes unnecessary spaces from the beginning and end of each text. A new column called `word_count` is then created by splitting each narrative into words and counting them. Complaints shorter than the minimum number of words defined in `CONFIG["min_words"]` are removed because very short texts usually contain too little information for meaningful clustering. Duplicate narratives are also removed to prevent repeated texts from having too much influence on the cluster centers. If the cleaned dataset is larger than `CONFIG["max_sample_size"]`, a reproducible random sample is created using the fixed `random_state`. Finally, the cell compares the number of rows before and after cleaning, confirms that no missing or duplicate narratives remain, and displays descriptive statistics for text length.

In [ ]:
# ============================================================
# Initial data cleaning
# ============================================================

df = df_raw.copy()


initial_row_count = len(df)


# Remove missing narratives
df = df.dropna(
    subset=[TEXT_COLUMN]
).copy()


# Convert narrative values to strings
df[TEXT_COLUMN] = (
    df[TEXT_COLUMN]
    .astype(str)
    .str.strip()
)


# Remove empty narratives
df = df[
    df[TEXT_COLUMN] != ""
].copy()


# Count words before filtering
df["word_count"] = (
    df[TEXT_COLUMN]
    .str.split()
    .str.len()
)


# Remove extremely short narratives
df = df[
    df["word_count"] >= CONFIG["min_words"]
].copy()


# Remove duplicate narratives
df = df.drop_duplicates(
    subset=[TEXT_COLUMN]
).reset_index(drop=True)


# Reproducible sampling if necessary
if len(df) > CONFIG["max_sample_size"]:

    df = df.sample(
        n=CONFIG["max_sample_size"],
        random_state=CONFIG["random_state"],
    ).reset_index(drop=True)


print("Initial rows:", initial_row_count)
print("Rows after cleaning:", len(df))

print("\nMissing narratives:")
print(df[TEXT_COLUMN].isna().sum())

print("\nDuplicate narratives:")
print(df[TEXT_COLUMN].duplicated().sum())

print("\nWord-count statistics:")
display(
    df["word_count"]
    .describe()
    .to_frame("value")
)

## Cell 7 — Text Preprocessing

Cell 7 defines a reusable function called `clean_text()` to clean each complaint narrative while preserving the words that are useful for topic discovery. The text is first converted to a string and transformed to lowercase with `str(text).lower()` so that words such as `Credit`, `CREDIT`, and `credit` are treated as the same feature. URLs are removed with `re.sub(r"https?://\S+|www\.\S+", " ", text)`, where `s?` makes the `s` in `https` optional, `\S+` matches one or more non-whitespace characters, `|` means “or,” and `www\.` matches URLs beginning with `www.`. Email addresses are removed with `re.sub(r"\S+@\S+", " ", text)`, which matches non-whitespace characters before and after the `@` symbol. Anonymization placeholders such as `XX`, `XXXX`, or `xxxx` are removed with `re.sub(r"\b[xX]{2,}\b", " ", text)`, where `\b` marks word boundaries, `[xX]` matches either uppercase or lowercase `x`, and `{2,}` means at least two repetitions. The expression `re.sub(r"[^a-zA-Z\s']", " ", text)` then removes numbers, punctuation, and special symbols while preserving English letters, whitespace, and apostrophes in words such as `don't` and `can't`. Repeated whitespace is reduced to a single space using `re.sub(r"\s+", " ", text)`, and `text.strip()` removes any remaining spaces from the beginning and end of the text. The letter `r` before each pattern means “raw string”; it does not mean removal, but ensures that backslashes such as `\s`, `\S`, and `\b` are passed correctly to the regular-expression engine. After the function is defined, `df[TEXT_COLUMN].apply(clean_text)` runs it on every complaint and stores the result in a new column called `clean_text`, while the original text remains unchanged. The code then calculates the number of words in each cleaned narrative and stores it in `clean_word_count`. Narratives that become shorter than `CONFIG["min_words"]` after cleaning are removed, because they may no longer contain enough meaningful information for clustering. Finally, the index is reset, the final dataset shape is printed, and the original text, cleaned text, and cleaned word count are displayed side by side for quality control. For example, the text `Contact me at USER@test.com or visit https://bank.com. My account XXXX was charged $500!!!` is transformed into `contact me at or visit my account was charged`, which is cleaner and more suitable for TF-IDF vectorization and K-means clustering.

In [ ]:
# ============================================================
# Text preprocessing
# ============================================================

def clean_text(text):
    """
    Apply light text cleaning while preserving useful words.
    """

    text = str(text).lower()

    # Remove URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text,
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text,
    )

    # Remove placeholder tokens used for anonymization
    text = re.sub(
        r"\b[xX]{2,}\b",
        " ",
        text,
    )

    # Keep letters, apostrophes and spaces
    text = re.sub(
        r"[^a-zA-Z\s']",
        " ",
        text,
    )

    # Remove repeated whitespace
    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()


df["clean_text"] = df[TEXT_COLUMN].apply(
    clean_text
)


# Recalculate word counts after cleaning
df["clean_word_count"] = (
    df["clean_text"]
    .str.split()
    .str.len()
)


# Remove texts that became too short after cleaning
df = df[
    df["clean_word_count"] >= CONFIG["min_words"]
].reset_index(drop=True)


print("Final cleaned dataset shape:")
print(df.shape)

display(
    df[
        [
            TEXT_COLUMN,
            "clean_text",
            "clean_word_count",
        ]
    ].head()
)

Cell 8 saves the cleaned DataFrame, df, to the path defined by CLEAN_DATA_PATH without including the internal row indices. It then displays the saved file path and the final number of remaining complaints. This file is the analysis-ready version of the dataset, and the subsequent vectorization and clustering steps are performed using these cleaned data.

In [ ]:
# ============================================================
# Save cleaned dataset
# ============================================================

df.to_csv(
    CLEAN_DATA_PATH,
    index=False,
)


print(f"Clean dataset saved to: {CLEAN_DATA_PATH}")
print("Final number of documents:", len(df))

> **Note on sample size:**  
> A final sample of 1,984 complaint narratives is still sufficient for this short K-means project. In unsupervised clustering, the main concern is not sample size alone, but whether the data contain enough thematic diversity and whether the resulting clusters are stable, balanced, and interpretable. For this dataset, using approximately 4–8 clusters would still leave a reasonable number of documents in each group. The analysis should therefore focus on cluster sizes, silhouette scores, top terms, representative documents, and stability across different random seeds rather than assuming that a larger dataset automatically produces better clusters.



In [ ]:
# ============================================================
# Basic exploratory analysis
# ============================================================

summary = pd.DataFrame(
    {
        "metric": [
            "Number of documents",
            "Average word count",
            "Median word count",
            "Minimum word count",
            "Maximum word count",
        ],
        "value": [
            len(df),
            round(df["clean_word_count"].mean(), 2),
            round(df["clean_word_count"].median(), 2),
            df["clean_word_count"].min(),
            df["clean_word_count"].max(),
        ],
    }
)


display(summary)


summary.to_csv(
    TABLE_DIR / "dataset_summary.csv",
    index=False,
)

## Cell 10 — Complaint Narrative Length Distribution

1. This chart is a histogram of complaint narrative lengths.  
2. The horizontal axis shows the number of words in each complaint.  
3. Moving to the right on the horizontal axis represents longer complaint narratives.  
4. The vertical axis shows the number of complaints within each word-count range.  
5. `bins=50` divides the full range of text lengths into 50 intervals.  
6. Bars located far to the right represent unusually long complaint narratives.  
7. The histogram helps identify the distribution shape, skewness, and possible outliers.

In [ ]:
# ============================================================
# Narrative length distribution
# ============================================================

plt.figure(figsize=(9, 5))

plt.hist(
    df["clean_word_count"],
    bins=50,
)

plt.xlabel("Number of words")
plt.ylabel("Number of complaints")
plt.title("Distribution of Complaint Narrative Lengths")
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "narrative_length_distribution.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## Cell 11 — TF-IDF Vectorization

Cell 11 converts the cleaned complaint narratives into numerical vectors because K-means cannot work directly with raw text. The `TfidfVectorizer` first identifies the vocabulary and then assigns a TF-IDF weight to each word or phrase based on how important it is within a document and how uncommon it is across the full collection. Common English stop words such as `the`, `is`, and `and` are removed with `stop_words="english"` because they usually do not help distinguish complaint topics. The parameter `min_df=5` removes terms that appear in fewer than five documents, while `max_df=0.90` removes terms that occur in more than 90% of the documents, reducing both rare noise and overly common words. The number of features is limited to a maximum of 8,000 through `max_features`, which helps control memory usage, runtime, and model complexity. The setting `ngram_range=(1, 2)` includes both individual words and two-word phrases, allowing the model to use features such as `credit`, `payment`, `credit report`, and `debt collection`. The option `norm="l2"` normalizes every document vector so that longer complaints do not receive greater influence simply because they contain more words. The setting `sublinear_tf=True` applies a logarithmic transformation to term frequency, reducing the excessive influence of words repeated many times within the same complaint. The method `fit_transform()` performs two operations: it learns the vocabulary and TF-IDF weights from `df["clean_text"]`, and then transforms every complaint into a numerical vector. The resulting sparse matrix, `X_tfidf`, stores one complaint per row and one word or phrase per column, while most entries remain zero because each document contains only a small subset of all possible features. The command `get_feature_names_out()` stores the selected words and phrases in `feature_names`, which will later be used to interpret the most important terms in each cluster. Finally, the cell prints the shape of the TF-IDF matrix, where the first value represents the number of documents and the second value represents the number of selected text features. The main output of this cell is `X_tfidf`, which becomes the numerical input for K-means clustering.

In [ ]:
# ============================================================
# TF-IDF vectorization
# ============================================================

vectorizer = TfidfVectorizer(
    stop_words="english",
    min_df=CONFIG["min_df"],
    max_df=CONFIG["max_df"],
    max_features=CONFIG["max_features"],
    ngram_range=CONFIG["ngram_range"],
    norm="l2",
    sublinear_tf=True,
)


X_tfidf = vectorizer.fit_transform(
    df["clean_text"]
)


feature_names = vectorizer.get_feature_names_out()


print("TF-IDF matrix shape:")
print(X_tfidf.shape)

print("\nNumber of documents:")
print(X_tfidf.shape[0])

print("\nNumber of TF-IDF features:")
print(X_tfidf.shape[1])

## Cell 12 — Evaluating Candidate Values of K

Cell 12 tests several possible numbers of clusters because K-means requires the value of `K` to be defined in advance. The loop evaluates values from `CONFIG["k_min"]` to `CONFIG["k_max"]`, which in this project means testing `K = 2` through `K = 10`. For each value of `K`, a new `KMeans` model is created with a fixed `random_state` for reproducibility, `n_init=20` so the algorithm is run from 20 different initial centroid positions, and `max_iter=300` to limit the maximum number of centroid-updating iterations. The method `fit_predict(X_tfidf)` trains the model on the TF-IDF matrix and assigns a cluster label to every complaint. The code then calculates the silhouette score, which measures how similar each document is to its own cluster compared with the nearest alternative cluster; values closer to 1 indicate better separation, values near 0 indicate overlapping clusters, and negative values may indicate incorrect assignments. The sample size used for the silhouette calculation is selected with `min()` so it never exceeds the number of available documents. The model’s inertia is also recorded; inertia measures the total squared distance between documents and their assigned cluster centers, so lower values indicate more compact clusters, although inertia naturally decreases as `K` increases. For every candidate value of `K`, the cluster count, inertia, and silhouette score are stored in `evaluation_results`. After the loop finishes, these results are converted into the DataFrame `evaluation_df`, displayed as a comparison table, and saved as `outputs/tables/k_evaluation_results.csv`. This table is later used to create the elbow and silhouette plots and to support the selection of an appropriate number of clusters.

In [ ]:
# ============================================================
# Evaluate candidate values of K
# ============================================================

evaluation_results = []


for k in range(
    CONFIG["k_min"],
    CONFIG["k_max"] + 1,
):

    print(f"Evaluating K = {k}")

    model = KMeans(
        n_clusters=k,
        random_state=CONFIG["random_state"],
        n_init=20,
        max_iter=300,
    )

    cluster_labels = model.fit_predict(
        X_tfidf
    )


    sample_size = min(
        CONFIG["silhouette_sample_size"],
        X_tfidf.shape[0],
    )


    silhouette = silhouette_score(
        X_tfidf,
        cluster_labels,
        sample_size=sample_size,
        random_state=CONFIG["random_state"],
    )


    evaluation_results.append(
        {
            "k": k,
            "inertia": model.inertia_,
            "silhouette_score": silhouette,
        }
    )


evaluation_df = pd.DataFrame(
    evaluation_results
)


display(evaluation_df)


evaluation_df.to_csv(
    TABLE_DIR / "k_evaluation_results.csv",
    index=False,
)

## Cell 13 — Elbow Plot

Cell 13 plots the inertia values of different K-means models against the number of clusters. The horizontal axis represents the number of clusters, while the vertical axis represents inertia. The purpose is to identify the point after which increasing the number of clusters no longer produces a substantial reduction in inertia. This point is known as the **elbow** and can be used to select an appropriate number of clusters.

In [ ]:
# ============================================================
# Elbow plot
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    evaluation_df["k"],
    evaluation_df["inertia"],
    marker="o",
)

plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-means")
plt.xticks(evaluation_df["k"])
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "elbow_plot.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## Cell 13 — Elbow Plot

Cell 13 plots the inertia values of different K-means models against the number of clusters. The horizontal axis represents the number of clusters, while the vertical axis represents inertia. The purpose is to identify the point after which increasing the number of clusters no longer produces a substantial reduction in inertia. This point is known as the **elbow** and can be used to select an appropriate number of clusters.

In [ ]:
# ============================================================
# Silhouette score plot
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    evaluation_df["k"],
    evaluation_df["silhouette_score"],
    marker="o",
)

plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette score")
plt.title("Silhouette Scores for Candidate K Values")
plt.xticks(evaluation_df["k"])
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "silhouette_scores.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## Cell 15 — Selecting the Best Number of Clusters

Cell 15 identifies the row in `evaluation_df` with the highest silhouette score. It then extracts the corresponding number of clusters and silhouette score and stores them in `best_k` and `best_silhouette`. The value of `best_k` is converted to an integer because it will be used as the number of clusters when training the final K-means model.

However, the final choice of `K` should not rely only on the highest silhouette score. Other criteria should also be examined, including the elbow plot, cluster sizes, balance between clusters, top terms, representative documents, and the overall interpretability of the cluster topics. A model may achieve the highest silhouette score while still producing very small, imbalanced, or difficult-to-interpret clusters.

In [ ]:
# ============================================================
# Select the best K
# ============================================================

BEST_K = int(
    evaluation_df.loc[
        evaluation_df["silhouette_score"].idxmax(),
        "k",
    ]
)


BEST_SILHOUETTE = float(
    evaluation_df["silhouette_score"].max()
)


print("Selected number of clusters:")
print(BEST_K)

print("\nBest silhouette score:")
print(round(BEST_SILHOUETTE, 4))

In [ ]:
# ============================================================
# Train final K-means model
# ============================================================

final_kmeans = KMeans(
    n_clusters=BEST_K,
    random_state=CONFIG["random_state"],
    n_init=20,
    max_iter=300,
)


df["cluster"] = final_kmeans.fit_predict(
    X_tfidf
)


print("Final K-means model trained successfully.")

print("\nCluster distribution:")
display(
    df["cluster"]
    .value_counts()
    .sort_index()
    .to_frame("document_count")
)

### Cluster Size Analysis

This cell analyses the distribution of documents across the clusters generated by the final K-means model. The cluster labels stored in the `cluster` column are counted using `value_counts()`, which determines the number of documents assigned to each cluster. The results are then sorted by cluster identifier to ensure that the clusters appear in ascending numerical order.

The index containing the cluster identifiers is renamed to `cluster` and converted into a regular DataFrame column. The frequency values are stored in a second column named `document_count`. This produces a structured table in which each row represents one cluster and reports the number of documents assigned to it.

A new column named `percentage` is calculated by dividing the number of documents in each cluster by the total number of documents in the dataset and multiplying the result by 100. The percentages are rounded to two decimal places to improve readability and make the table suitable for reporting.

The resulting table contains three variables:

* `cluster`: the numerical identifier of the cluster.
* `document_count`: the number of documents assigned to the cluster.
* `percentage`: the proportion of the complete dataset represented by the cluster.

The table is displayed in the notebook and exported as `cluster_sizes.csv` in the designated tables directory. The DataFrame index is excluded from the exported file because it does not contain meaningful analytical information.

Cluster size analysis provides an initial overview of how the K-means model partitioned the dataset. Large clusters may represent broad and frequently occurring topics, whereas smaller clusters may capture more specific or less common themes. However, balanced cluster sizes do not necessarily indicate high-quality clustering, and unequal cluster sizes are not automatically problematic. The distribution should therefore be interpreted together with the silhouette score, the top terms associated with each cluster, and a qualitative examination of representative documents.


In [ ]:
# ============================================================
# Cluster size analysis
# ============================================================

cluster_sizes = (
    df["cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("cluster")
    .reset_index(name="document_count")
)


cluster_sizes["percentage"] = (
    cluster_sizes["document_count"]
    / len(df)
    * 100
).round(2)


display(cluster_sizes)


cluster_sizes.to_csv(
    TABLE_DIR / "cluster_sizes.csv",
    index=False,
)

### Extracting Top Terms from Each Cluster

This cell extracts the most important terms associated with each cluster. The cluster centres are obtained from the trained K-means model, and the features with the highest centroid weights are selected.

The feature indices are converted back into their corresponding terms using `feature_names`. Each term is stored together with its cluster number and rank.

The results are converted into a DataFrame and saved as `cluster_top_terms.csv`. These terms help identify the main topic represented by each cluster.


In [ ]:
# ============================================================
# Extract top terms for each cluster
# ============================================================

cluster_centers = final_kmeans.cluster_centers_


top_terms_records = []


for cluster_id in range(BEST_K):

    top_indices = (
        cluster_centers[cluster_id]
        .argsort()[::-1][
            :CONFIG["top_terms_per_cluster"]
        ]
    )


    top_terms = feature_names[top_indices]


    print(f"\nCluster {cluster_id}")
    print(", ".join(top_terms))


    for rank, term in enumerate(
        top_terms,
        start=1,
    ):

        top_terms_records.append(
            {
                "cluster": cluster_id,
                "rank": rank,
                "term": term,
            }
        )


top_terms_df = pd.DataFrame(
    top_terms_records
)


top_terms_df.to_csv(
    TABLE_DIR / "cluster_top_terms.csv",
    index=False,
)

### Identifying Representative Documents

This cell identifies the documents that are closest to the centre of each cluster. These documents can be considered representative examples because their TF-IDF vectors are highly similar to the overall characteristics of their assigned cluster.

First, the trained K-means model calculates the distance of every document from every cluster centre. For each cluster, the code selects the documents assigned to that cluster and retrieves their distances from the corresponding centroid.

The distances are sorted in ascending order, and the configured number of closest documents is selected. A smaller distance indicates that a document is more representative of the cluster.

For each selected document, the cluster number, ranking, and complete narrative are stored. Product and issue information are also included when these columns are available. The first 1,000 characters of each narrative are printed to keep the notebook output readable.

Finally, the representative documents are converted into a DataFrame and saved as `representative_complaints.csv`. Examining these documents supports the qualitative interpretation and manual labelling of the clusters.


In [ ]:
# ============================================================
# Find representative documents
# ============================================================

distances = final_kmeans.transform(
    X_tfidf
)


representative_records = []


for cluster_id in range(BEST_K):

    cluster_indices = np.where(
        df["cluster"].to_numpy() == cluster_id
    )[0]


    cluster_distances = distances[
        cluster_indices,
        cluster_id,
    ]


    closest_positions = np.argsort(
        cluster_distances
    )[
        :CONFIG["representative_documents"]
    ]


    representative_indices = cluster_indices[
        closest_positions
    ]


    print("\n" + "=" * 90)
    print(f"CLUSTER {cluster_id}")
    print("=" * 90)


    for rank, row_index in enumerate(
        representative_indices,
        start=1,
    ):

        narrative = df.loc[
            row_index,
            TEXT_COLUMN,
        ]


        print(f"\nRepresentative document {rank}:")
        print(narrative[:1_000])


        record = {
            "cluster": cluster_id,
            "rank": rank,
            "narrative": narrative,
        }


        if PRODUCT_COLUMN is not None:
            record["product"] = df.loc[
                row_index,
                PRODUCT_COLUMN,
            ]


        if ISSUE_COLUMN is not None:
            record["issue"] = df.loc[
                row_index,
                ISSUE_COLUMN,
            ]


        representative_records.append(record)


representative_df = pd.DataFrame(
    representative_records
)


representative_df.to_csv(
    TABLE_DIR / "representative_complaints.csv",
    index=False,
)

### Product Distribution by Cluster

This cell analyzes the distribution of products within each cluster.

If a valid product column is available, the code:

1. Groups the dataset by cluster and product.
2. Counts how many times each product appears in each cluster.
3. Ranks the products within each cluster based on their frequency.
4. Selects the five most frequent products from every cluster.
5. Sorts the results by cluster number and product rank.
6. Displays the final table.
7. Saves the results as:

`top_products_by_cluster.csv`

If no product column is available, the analysis is skipped and an explanatory message is printed.


In [ ]:
# ============================================================
# Product distribution by cluster
# ============================================================

if PRODUCT_COLUMN is not None:

    cluster_product_distribution = (
        df.groupby(
            ["cluster", PRODUCT_COLUMN]
        )
        .size()
        .reset_index(name="count")
    )


    cluster_product_distribution["rank"] = (
        cluster_product_distribution
        .groupby("cluster")["count"]
        .rank(
            method="first",
            ascending=False,
        )
    )


    top_products_by_cluster = (
        cluster_product_distribution[
            cluster_product_distribution["rank"] <= 5
        ]
        .sort_values(
            ["cluster", "rank"]
        )
    )


    display(top_products_by_cluster)


    top_products_by_cluster.to_csv(
        TABLE_DIR / "top_products_by_cluster.csv",
        index=False,
    )

else:

    print(
        "No product column was available. "
        "This analysis was skipped."
    )

In [ ]:
# ============================================================
# Issue distribution by cluster
# ============================================================

if ISSUE_COLUMN is not None:

    cluster_issue_distribution = (
        df.groupby(
            ["cluster", ISSUE_COLUMN]
        )
        .size()
        .reset_index(name="count")
    )


    cluster_issue_distribution["rank"] = (
        cluster_issue_distribution
        .groupby("cluster")["count"]
        .rank(
            method="first",
            ascending=False,
        )
    )


    top_issues_by_cluster = (
        cluster_issue_distribution[
            cluster_issue_distribution["rank"] <= 5
        ]
        .sort_values(
            ["cluster", "rank"]
        )
    )


    display(top_issues_by_cluster)


    top_issues_by_cluster.to_csv(
        TABLE_DIR / "top_issues_by_cluster.csv",
        index=False,
    )

else:

    print(
        "No issue column was available. "
        "This analysis was skipped."
    )

### Two-Dimensional Representation

This cell reduces the high-dimensional TF-IDF matrix to two dimensions using Truncated Singular Value Decomposition.

The code:

1. Creates a `TruncatedSVD` model with two components.
2. Transforms the TF-IDF feature matrix into a two-dimensional representation.
3. Stores the two components and the assigned cluster of each document in a new DataFrame.
4. Calculates the total proportion of variance explained by the two components.
5. Prints the explained variance rounded to four decimal places.

The resulting `visualization_df` can be used to visualize the documents and their cluster assignments in a two-dimensional plot.


In [ ]:
# ============================================================
# Two-dimensional representation
# ============================================================

svd = TruncatedSVD(
    n_components=2,
    random_state=CONFIG["random_state"],
)


X_2d = svd.fit_transform(
    X_tfidf
)


visualization_df = pd.DataFrame(
    {
        "component_1": X_2d[:, 0],
        "component_2": X_2d[:, 1],
        "cluster": df["cluster"],
    }
)


print("Explained variance by two components:")
print(
    round(
        svd.explained_variance_ratio_.sum(),
        4,
    )
)

The two-dimensional SVD visualization shows substantial overlap among most clusters, indicating that the cluster boundaries are not clearly visible in the reduced two-dimensional space. However, a few small groups appear relatively distinct. Since the visualization preserves only part of the variance in the original TF-IDF space, the overlap does not necessarily imply poor clustering performance and should be interpreted together with quantitative metrics such as the silhouette score.


In [ ]:
# ============================================================
# Cluster visualization
# ============================================================

plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    visualization_df["component_1"],
    visualization_df["component_2"],
    c=visualization_df["cluster"],
    alpha=0.50,
    s=12,
)

plt.xlabel("SVD component 1")
plt.ylabel("SVD component 2")
plt.title("Two-Dimensional Visualization of K-means Clusters")
plt.colorbar(
    scatter,
    label="Cluster",
)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "cluster_visualization.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ============================================================
# Save final clustered dataset
# ============================================================

FINAL_DATA_PATH = OUTPUT_DIR / "clustered_complaints.csv"


df.to_csv(
    FINAL_DATA_PATH,
    index=False,
)


print(f"Clustered dataset saved to: {FINAL_DATA_PATH}")

In [ ]:
# ============================================================
# Create project summary
# ============================================================

project_summary = {
    "dataset": CONFIG["dataset_name"],
    "number_of_documents": int(len(df)),
    "text_column": TEXT_COLUMN,
    "product_column": PRODUCT_COLUMN,
    "issue_column": ISSUE_COLUMN,
    "tfidf_features": int(X_tfidf.shape[1]),
    "selected_k": int(BEST_K),
    "best_silhouette_score": round(
        BEST_SILHOUETTE,
        4,
    ),
    "random_state": CONFIG["random_state"],
}


SUMMARY_PATH = OUTPUT_DIR / "project_summary.json"


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        project_summary,
        file,
        indent=4,
    )


print("Project summary:")
print(
    json.dumps(
        project_summary,
        indent=4,
    )
)

In [ ]:
# ============================================================
# Final pipeline validation
# ============================================================

required_output_files = [
    CLEAN_DATA_PATH,
    OUTPUT_DIR / "clustered_complaints.csv",
    OUTPUT_DIR / "project_summary.json",
    TABLE_DIR / "k_evaluation_results.csv",
    TABLE_DIR / "cluster_sizes.csv",
    TABLE_DIR / "cluster_top_terms.csv",
    TABLE_DIR / "representative_complaints.csv",
    FIGURE_DIR / "elbow_plot.png",
    FIGURE_DIR / "silhouette_scores.png",
    FIGURE_DIR / "cluster_visualization.png",
]


missing_output_files = [
    str(path)
    for path in required_output_files
    if not path.exists()
]


if missing_output_files:

    raise RuntimeError(
        "Pipeline completed with missing outputs: "
        + ", ".join(missing_output_files)
    )


print("=" * 70)
print("END-TO-END PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 70)

print(f"Documents analysed: {len(df):,}")
print(f"Selected K: {BEST_K}")
print(
    f"Silhouette score: "
    f"{BEST_SILHOUETTE:.4f}"
)
print(f"Outputs directory: {OUTPUT_DIR.resolve()}")

In [ ]:
# ============================================================
# Compress and download all output files
# ============================================================

from pathlib import Path
import shutil


OUTPUT_DIR = Path(OUTPUT_DIR)

ZIP_NAME = "kmeans_clustering_outputs"
ZIP_PATH = Path(f"{ZIP_NAME}.zip")


if not OUTPUT_DIR.exists():

    raise FileNotFoundError(
        f"Output directory was not found: {OUTPUT_DIR}"
    )


zip_file = shutil.make_archive(
    base_name=ZIP_NAME,
    format="zip",
    root_dir=OUTPUT_DIR.parent,
    base_dir=OUTPUT_DIR.name,
)


print(f"ZIP file created successfully: {zip_file}")


try:

    from google.colab import files

    files.download(zip_file)

except ImportError:

    from IPython.display import FileLink, display

    print("Click the link below to download the ZIP file:")

    display(
        FileLink(zip_file)
    )